# BVP, EDA, BVP+EDA Personalized, LOSO

## 1) Personalized

In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pointbiserialr, spearmanr

import warnings, os
warnings.filterwarnings('ignore')

# ── 파일 경로 ──
BPV_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP_TPV/noQC_BVP.csv"
EDA_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/EDA_TPV/noQC_EDA.csv"

# ── 데이터 로드 ──
sep_bpv = '\t' if open(BPV_PATH).readline().count('\t') > 3 else ','
sep_eda = '\t' if open(EDA_PATH).readline().count('\t') > 3 else ','

df_bpv = pd.read_csv(BPV_PATH, sep=sep_bpv)
df_eda = pd.read_csv(EDA_PATH, sep=sep_eda)

df_bpv.columns = df_bpv.columns.str.strip()
df_eda.columns = df_eda.columns.str.strip()

print(f"BPV shape: {df_bpv.shape}, EDA shape: {df_eda.shape}")
print(f"BPV columns: {list(df_bpv.columns)}")
print(f"EDA columns: {list(df_eda.columns)}")

# ══════════════════════════════════════════════════════════════════════════════
# EmpaticaE4Stress 이진 분류 라벨 처리
# label_major: normal(0), stress(1)
# 기존 로직 유지 위해 0/1만 남기고 그대로 사용
# ══════════════════════════════════════════════════════════════════════════════
LABEL_MAP = {0: 0, 1: 1}
LABEL_NAMES = {0: 'Normal', 1: 'Stress'}

def to_binary(df, label_col='label_major'):
    df = df[df[label_col].isin(LABEL_MAP.keys())].copy()
    df[label_col] = df[label_col].map(LABEL_MAP)
    return df

df_bpv = to_binary(df_bpv)
df_eda = to_binary(df_eda)

print(f"\n[이진 분류] label_major 매핑: {LABEL_MAP}")
print(f"  BPV after filter: {df_bpv.shape}, class dist: {df_bpv['label_major'].value_counts().to_dict()}")
print(f"  EDA after filter: {df_eda.shape}, class dist: {df_eda['label_major'].value_counts().to_dict()}")

# ── TPV 피처 컬럼 식별 ──
meta_cols = [
    'subject', 'window', 'window_index', 'status', 'status_name',
    'label_major', 'phase', 't_start_sec', 't_end_sec', 'major_ratio',
    'subject_total_duration_sec', 'task4_duration_sec',
    'bvp_missing_ratio', 'acc_missing_ratio', 'missing_flag',
    'flatline_flag', 'clipping_ratio', 'clipping_flag', 'n_peaks',
    'peak_sparse_flag', 'ibi_pass_ratio', 'valid_beat_ratio',
    'ibi_plausibility', 'ibi_mean_sec', 'median_template_corr',
    'notch_foot_peak_detectability', 'acc_diff_mean',
    'acc_diff_exceed_ratio', 'acc_diff_flag', 'psd_corr_x',
    'psd_corr_y', 'psd_corr_z', 'psd_corr_mag', 'psd_corr_max',
    'psd_corr_flag', 'motion_score_raw', 'motion_score',
    'SQI_stress_basic', 'SQI_stress_morph', 'hard_reject',
    'motion_threshold_subject_p80', 'qc_relaxed_pass'
]

bpv_feat_cols = [c for c in df_bpv.columns if c not in meta_cols]
eda_feat_cols = [c for c in df_eda.columns if c not in meta_cols]

# EDA에서 사용할 선택된 TPV feature만 유지
selected_eda_tpv = [
    'TPV27', 'TPV1', 'TPV12', 'TPV16', 'TPV0', 'TPV2', 'TPV3', 'TPV4',
    'TPV9', 'TPV15', 'TPV17', 'TPV20', 'TPV21', 'TPV22', 'TPV24', 'TPV5',
    'TPV6', 'TPV8', 'TPV10', 'TPV11', 'TPV23', 'TPV28', 'TPV30', 'TPV31'
]

eda_feat_cols = [c for c in selected_eda_tpv if c in eda_feat_cols]

print(f"\nBPV feature cols ({len(bpv_feat_cols)}): {bpv_feat_cols}")
print(f"EDA feature cols ({len(eda_feat_cols)}): {eda_feat_cols}")

# ── 머지 (BPV+EDA) ──
merge_keys = ['subject', 'window_index']
df_merged = pd.merge(
    df_bpv[merge_keys + ['label_major'] + bpv_feat_cols],
    df_eda[merge_keys + eda_feat_cols],
    on=merge_keys, how='inner', suffixes=('_bpv', '_eda')
)

bpv_feat_merged = [
    c for c in df_merged.columns
    if c not in merge_keys + ['label_major']
    and (c in bpv_feat_cols or c.endswith('_bpv'))
]

eda_feat_merged = [
    c for c in df_merged.columns
    if c not in merge_keys + ['label_major']
    and (c in eda_feat_cols or c.endswith('_eda'))
    and c not in bpv_feat_merged
]

combined_feat = bpv_feat_merged + eda_feat_merged

df_bpv_only = df_bpv[merge_keys + ['label_major'] + bpv_feat_cols].copy()
df_eda_only = df_eda[merge_keys + ['label_major'] + eda_feat_cols].copy()

print(f"\nMerged shape: {df_merged.shape}")
print(f"Merged class dist: {df_merged['label_major'].value_counts().to_dict()}")
print(f"BPV merged feats ({len(bpv_feat_merged)}): {bpv_feat_merged[:10]}...")
print(f"EDA merged feats ({len(eda_feat_merged)}): {eda_feat_merged[:10]}...")
print(f"Combined feats (before multicollinearity removal): {len(combined_feat)}")

# ══════════════════════════════════════════════════════════════════════════════
# 다중 공선성 제거
# ══════════════════════════════════════════════════════════════════════════════
def remove_high_corr_features(df, feat_cols, threshold=0.85):
    corr_mat = df[feat_cols].astype(float).corr(method='spearman').abs()

    label_corr = {}
    for c in feat_cols:
        vals = df[[c, 'label_major']].dropna()
        if len(vals) < 5:
            label_corr[c] = 0.0
        else:
            r, _ = spearmanr(vals[c].astype(float), vals['label_major'].astype(float))
            label_corr[c] = abs(r)

    to_drop = set()
    for i in range(len(feat_cols)):
        if feat_cols[i] in to_drop:
            continue
        for j in range(i + 1, len(feat_cols)):
            if feat_cols[j] in to_drop:
                continue
            if corr_mat.iloc[i, j] > threshold:
                fi, fj = feat_cols[i], feat_cols[j]
                if label_corr[fi] >= label_corr[fj]:
                    to_drop.add(fj)
                else:
                    to_drop.add(fi)

    kept = [c for c in feat_cols if c not in to_drop]
    return kept, to_drop

def compute_vif_and_filter(df, feat_cols, vif_threshold=10.0, max_iter=50):
    remaining = list(feat_cols)
    X = df[remaining].astype(float).dropna()

    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=remaining)

    dropped_vif = []
    for iteration in range(max_iter):
        if len(remaining) <= 2:
            break

        X_cur = X_scaled[remaining].values
        vifs = []
        for k in range(len(remaining)):
            try:
                v = variance_inflation_factor(X_cur, k)
            except:
                v = np.inf
            vifs.append(v)

        max_vif = max(vifs)
        if max_vif <= vif_threshold:
            break

        worst_idx = np.argmax(vifs)
        worst_feat = remaining[worst_idx]
        dropped_vif.append((worst_feat, max_vif))
        remaining.pop(worst_idx)

    return remaining, dropped_vif

def remove_multicollinearity(df, feat_cols, corr_threshold=0.85, vif_threshold=10.0):
    print(f"\n{'='*70}")
    print(f"  다중 공선성 제거 (corr_threshold={corr_threshold}, VIF_threshold={vif_threshold})")
    print(f"{'='*70}")
    print(f"  시작 피처 수: {len(feat_cols)}")

    kept_corr, dropped_corr = remove_high_corr_features(df, feat_cols, threshold=corr_threshold)
    print(f"\n  [Step 1] Spearman |r| > {corr_threshold} 제거: {len(dropped_corr)}개 제거")
    if dropped_corr:
        for d in sorted(dropped_corr):
            print(f"    - {d}")
    print(f"  → 남은 피처: {len(kept_corr)}개")

    kept_vif, dropped_vif = compute_vif_and_filter(df, kept_corr, vif_threshold=vif_threshold)
    print(f"\n  [Step 2] VIF > {vif_threshold} 반복 제거: {len(dropped_vif)}개 제거")
    if dropped_vif:
        for feat, vif_val in dropped_vif:
            print(f"    - {feat} (VIF={vif_val:.1f})")
    print(f"  → 최종 피처: {len(kept_vif)}개")

    if len(kept_vif) >= 2:
        X_final = df[kept_vif].astype(float).dropna()
        scaler = StandardScaler()
        X_sc = scaler.fit_transform(X_final)
        final_vifs = []
        for k in range(len(kept_vif)):
            try:
                v = variance_inflation_factor(X_sc, k)
            except:
                v = np.inf
            final_vifs.append(v)
        print(f"\n  최종 VIF 확인:")
        for feat, vif_val in zip(kept_vif, final_vifs):
            print(f"    {feat:30s} VIF={vif_val:.2f}")

    total_removed = len(feat_cols) - len(kept_vif)
    print(f"\n  총 제거: {total_removed}개 ({len(feat_cols)} → {len(kept_vif)})")
    return kept_vif

combined_feat_clean = remove_multicollinearity(
    df_merged, combined_feat, corr_threshold=0.85, vif_threshold=10.0
)

# ══════════════════════════════════════════════════════════════════════════════
# 상관관계 분석
# ══════════════════════════════════════════════════════════════════════════════
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/outputs_empatica_e4stress"
os.makedirs(OUT_DIR, exist_ok=True)

def plot_feature_label_corr(df, feat_cols, label_col, title, fname):
    corrs = []
    for c in feat_cols:
        vals = df[[c, label_col]].dropna()
        if len(vals) < 5:
            continue
        x, y = vals[c].astype(float), vals[label_col].astype(float)
        try:
            pb_r, pb_p = pointbiserialr(y, x)
        except:
            pb_r, pb_p = np.nan, np.nan
        sp_r, sp_p = spearmanr(x, y)
        corrs.append({
            'Feature': c,
            'PointBiserial_r': pb_r,
            'PB_pval': pb_p,
            'Spearman_r': sp_r,
            'SP_pval': sp_p
        })

    df_corr = pd.DataFrame(corrs).sort_values('Spearman_r', key=abs, ascending=False)

    print(f"\n{'─'*70}")
    print(f"  {title}: Feature-Label Correlation (top 15) [Binary: Normal vs Stress]")
    print(f"{'─'*70}")
    print(df_corr.head(15).to_string(index=False))

    top = df_corr.head(20)
    fig, ax = plt.subplots(figsize=(10, max(4, len(top) * 0.35)))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top['Spearman_r']]
    ax.barh(range(len(top)), top['Spearman_r'].values, color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Feature'].values, fontsize=8)
    ax.set_xlabel('Spearman r')
    ax.set_title(f'{title} — Feature-Label Spearman Correlation (Binary)')
    ax.axvline(0, color='k', linewidth=0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_corr

def plot_feature_corr_matrix(df, feat_cols, title, fname):
    corr_mat = df[feat_cols].astype(float).corr(method='spearman')

    fig, ax = plt.subplots(
        figsize=(max(8, len(feat_cols) * 0.5), max(6, len(feat_cols) * 0.4))
    )
    mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
    sns.heatmap(
        corr_mat, mask=mask, cmap='RdBu_r', center=0,
        vmin=-1, vmax=1, annot=len(feat_cols) <= 20,
        fmt='.2f' if len(feat_cols) <= 20 else '',
        square=True, linewidths=0.5,
        cbar_kws={'shrink': 0.8}, ax=ax
    )
    ax.set_title(f'{title} — Feature Correlation Matrix (Spearman)', fontsize=11)
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()

    pairs = []
    for i in range(len(feat_cols)):
        for j in range(i + 1, len(feat_cols)):
            r = corr_mat.iloc[i, j]
            if abs(r) > 0.7:
                pairs.append((feat_cols[i], feat_cols[j], r))

    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    if pairs:
        print(f"\n  {title}: 높은 피처간 상관 (|r|>0.7) — {len(pairs)}쌍")
        for a, b, r in pairs[:15]:
            print(f"    {a:20s} ↔ {b:20s} : r={r:.4f}")
    return corr_mat

def plot_cross_corr_bpv_eda(df_merged, bpv_feats, eda_feats, fname):
    all_feats = bpv_feats + eda_feats
    corr_full = df_merged[all_feats].astype(float).corr(method='spearman')
    cross = corr_full.loc[bpv_feats, eda_feats]

    fig, ax = plt.subplots(
        figsize=(max(8, len(eda_feats) * 0.5), max(5, len(bpv_feats) * 0.4))
    )
    sns.heatmap(
        cross, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        annot=max(len(bpv_feats), len(eda_feats)) <= 20,
        fmt='.2f' if max(len(bpv_feats), len(eda_feats)) <= 20 else '',
        linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax
    )
    ax.set_title('BPV vs EDA — Cross-Feature Correlation (Spearman, Binary)', fontsize=11)
    ax.set_ylabel('BPV Features')
    ax.set_xlabel('EDA Features')
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()

    print(f"\n{'─'*70}")
    print(f"  BPV ↔ EDA Cross-Correlation (top 15 by |r|)")
    print(f"{'─'*70}")
    cross_pairs = []
    for b in bpv_feats:
        for e in eda_feats:
            cross_pairs.append((b, e, cross.loc[b, e]))
    cross_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    for a, b, r in cross_pairs[:15]:
        print(f"    {a:20s} ↔ {b:20s} : r={r:.4f}")
    return cross

def plot_subject_corr_comparison(df_src, feat_cols, label_col, title, fname):
    subjects = sorted(df_src['subject'].unique())
    corr_dict = {}
    for subj in subjects:
        ds = df_src[df_src['subject'] == subj]
        rs = []
        for c in feat_cols:
            vals = ds[[c, label_col]].dropna()
            if len(vals) < 5:
                rs.append(np.nan)
            else:
                r, _ = spearmanr(vals[c].astype(float), vals[label_col].astype(float))
                rs.append(r)
        corr_dict[subj] = rs

    df_subj_corr = pd.DataFrame(corr_dict, index=feat_cols).T

    fig, ax = plt.subplots(
        figsize=(max(8, len(feat_cols) * 0.5), max(4, len(subjects) * 0.4))
    )
    sns.heatmap(
        df_subj_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        annot=len(feat_cols) <= 15 and len(subjects) <= 20,
        fmt='.2f' if len(feat_cols) <= 15 else '',
        linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax
    )
    ax.set_title(f'{title} — Subject-wise Feature-Label Correlation (Binary)', fontsize=11)
    ax.set_ylabel('Subject')
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_subj_corr

print(f"\n\n{'='*80}")
print("  상관관계 분석 (Binary: Normal vs Stress)")
print(f"{'='*80}")

corr_bpv_label = plot_feature_label_corr(
    df_bpv_only, bpv_feat_cols, 'label_major', 'BPV', 'corr_bpv_label.png'
)

corr_eda_label = plot_feature_label_corr(
    df_eda_only, eda_feat_cols, 'label_major', 'EDA', 'corr_eda_label.png'
)

corr_comb_label_raw = plot_feature_label_corr(
    df_merged, combined_feat, 'label_major', 'BPV+EDA (raw)', 'corr_combined_label_raw.png'
)

corr_comb_label = plot_feature_label_corr(
    df_merged, combined_feat_clean, 'label_major', 'BPV+EDA (clean)', 'corr_combined_label.png'
)

plot_feature_corr_matrix(df_bpv_only, bpv_feat_cols, 'BPV', 'corr_matrix_bpv.png')
plot_feature_corr_matrix(df_eda_only, eda_feat_cols, 'EDA', 'corr_matrix_eda.png')
plot_feature_corr_matrix(df_merged, combined_feat_clean, 'BPV+EDA (clean)', 'corr_matrix_combined_clean.png')

plot_cross_corr_bpv_eda(
    df_merged, bpv_feat_merged, eda_feat_merged, 'corr_cross_bpv_eda.png'
)

subj_corr_bpv = plot_subject_corr_comparison(
    df_bpv_only, bpv_feat_cols, 'label_major', 'BPV', 'corr_subject_bpv.png'
)

subj_corr_eda = plot_subject_corr_comparison(
    df_eda_only, eda_feat_cols, 'label_major', 'EDA', 'corr_subject_eda.png'
)

corr_bpv_label.to_csv(os.path.join(OUT_DIR, 'corr_bpv_label.csv'), index=False)
corr_eda_label.to_csv(os.path.join(OUT_DIR, 'corr_eda_label.csv'), index=False)
corr_comb_label.to_csv(os.path.join(OUT_DIR, 'corr_combined_label.csv'), index=False)

print(f"\n상관분석 그래프 저장 완료 → {OUT_DIR}/corr_*.png")
print(f"상관계수 CSV 저장 완료 → {OUT_DIR}/corr_*_label.csv")

# ── 분류기 정의 ──
def get_classifiers():
    return {
        'RF': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'DT': DecisionTreeClassifier(random_state=42),
        'SVC': SVC(kernel='rbf', probability=True, random_state=42),
        'LR': LogisticRegression(max_iter=2000, random_state=42)
    }

# ── 평가 함수 ──
def evaluate(clf, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)

    try:
        y_prob = clf.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred)
    return acc, f1, auc, cm, y_pred

# ── 실험 실행 ──
feature_configs = {
    'BPV': (df_bpv_only, bpv_feat_cols),
    'EDA': (df_eda_only, eda_feat_cols),
    'BPV+EDA': (df_merged, combined_feat_clean),
}

all_results = []

subjects = sorted(df_merged['subject'].unique())
print(f"\n{'='*80}")
print(f"총 {len(subjects)} 명 피험자: {subjects}")
print(f"이진 분류: Normal(0) vs Stress(1)")
print(f"{'='*80}\n")

for feat_name, (df_src, feat_cols) in feature_configs.items():
    print(f"\n{'#'*80}")
    print(f"  Feature Set: {feat_name} ({len(feat_cols)} features)")
    print(f"{'#'*80}")

    for subj in subjects:
        df_subj = df_src[df_src['subject'] == subj].copy()
        df_subj = df_subj.dropna(subset=feat_cols + ['label_major'])

        X = df_subj[feat_cols].values.astype(float)
        y = df_subj['label_major'].values.astype(int)

        n_classes = len(np.unique(y))
        if len(df_subj) < 5 or n_classes < 2:
            print(f"\n  [{subj}] 샘플 부족 (n={len(df_subj)}, classes={n_classes}) → SKIP")
            continue

        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.3, random_state=42, stratify=y
        )

        print(f"\n  ── {subj} | Train={len(X_train)}, Test={len(X_test)}, "
              f"Class dist: {dict(zip(*np.unique(y, return_counts=True)))} ──")

        for clf_name, clf in get_classifiers().items():
            acc, f1, auc, cm, y_pred = evaluate(clf, X_train, X_test, y_train, y_test)

            all_results.append({
                'Feature': feat_name,
                'Subject': subj,
                'Classifier': clf_name,
                'Accuracy': acc,
                'F1_binary': f1,
                'AUC': auc,
            })

            print(f"    {clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | CM={cm.tolist()}")

# ── 결과 요약 ──
df_res = pd.DataFrame(all_results)

print(f"\n\n{'='*80}")
print("  전체 결과 요약 — Binary Classification (Normal vs Stress)")
print(f"{'='*80}")

for feat_name in ['BPV', 'EDA', 'BPV+EDA']:
    sub = df_res[df_res['Feature'] == feat_name]
    if sub.empty:
        continue

    print(f"\n── {feat_name} ──")
    pivot_acc = sub.pivot_table(index='Subject', columns='Classifier', values='Accuracy', aggfunc='first')
    pivot_f1 = sub.pivot_table(index='Subject', columns='Classifier', values='F1_binary', aggfunc='first')
    pivot_auc = sub.pivot_table(index='Subject', columns='Classifier', values='AUC', aggfunc='first')

    print("\n[Accuracy]")
    print(pivot_acc.round(4).to_string())
    print(f"  Mean: {pivot_acc.mean().round(4).to_dict()}")

    print("\n[F1 binary]")
    print(pivot_f1.round(4).to_string())
    print(f"  Mean: {pivot_f1.mean().round(4).to_dict()}")

    print("\n[AUC]")
    print(pivot_auc.round(4).to_string())
    print(f"  Mean: {pivot_auc.mean().round(4).to_dict()}")

# ── CSV 저장 ──
OUT_PATH = "/content/drive/MyDrive/Colab Notebooks/personalized_results_binary_empatica_e4stress.csv"
df_res.to_csv(OUT_PATH, index=False)
print(f"\n결과 저장: {OUT_PATH}")

# ── Feature set 간 평균 비교 ──
print(f"\n\n{'='*80}")
print("  Feature Set 간 평균 성능 비교 (Binary)")
print(f"{'='*80}")
summary = df_res.groupby(['Feature', 'Classifier'])[['Accuracy', 'F1_binary', 'AUC']].mean()
print(summary.round(4).to_string())

# ── 제거된 피처 정보 저장 ──
removed_feats = [c for c in combined_feat if c not in combined_feat_clean]
df_removed = pd.DataFrame({
    'removed_feature': removed_feats,
    'source': ['BPV' if c in bpv_feat_merged else 'EDA' for c in removed_feats]
})
df_removed.to_csv(os.path.join(OUT_DIR, 'removed_multicollinear_features.csv'), index=False)

print(f"\n제거된 피처 목록 저장: {OUT_DIR}/removed_multicollinear_features.csv")
print(f"제거된 피처 수: {len(removed_feats)}, 최종 BPV+EDA 피처 수: {len(combined_feat_clean)}")

BPV shape: (1035, 53), EDA shape: (516, 74)
BPV columns: ['subject', 'window', 'window_index', 'status', 'status_name', 'label_major', 'phase', 't_start_sec', 't_end_sec', 'major_ratio', 'subject_total_duration_sec', 'task4_duration_sec', 'bvp_missing_ratio', 'acc_missing_ratio', 'missing_flag', 'flatline_flag', 'clipping_ratio', 'clipping_flag', 'n_peaks', 'peak_sparse_flag', 'ibi_pass_ratio', 'valid_beat_ratio', 'ibi_plausibility', 'ibi_mean_sec', 'median_template_corr', 'notch_foot_peak_detectability', 'acc_diff_mean', 'acc_diff_exceed_ratio', 'acc_diff_flag', 'psd_corr_x', 'psd_corr_y', 'psd_corr_z', 'psd_corr_mag', 'psd_corr_max', 'psd_corr_flag', 'motion_score_raw', 'motion_score', 'SQI_stress_basic', 'SQI_stress_morph', 'hard_reject', 'motion_threshold_subject_p80', 'qc_relaxed_pass', 'TPV22', 'TPV9', 'TPV4', 'TPV2', 'TPV0', 'TPV28', 'TPV8', 'TPV19', 'TPV14', 'TPV26', 'TPV11']
EDA columns: ['subject', 'window', 'window_index', 'status', 'status_name', 'label_major', 'phase', 't_

## 2) LOSO

In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score,
                             confusion_matrix)
from sklearn.preprocessing import StandardScaler
import warnings, os
warnings.filterwarnings('ignore')

# ── 파일 경로 ──
BPV_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP_TPV/noQC_BVP.csv"
EDA_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/EDA_TPV/noQC_EDA.csv"

# ── 데이터 로드 ──
sep_bpv = '\t' if open(BPV_PATH).readline().count('\t') > 3 else ','
sep_eda = '\t' if open(EDA_PATH).readline().count('\t') > 3 else ','
df_bpv = pd.read_csv(BPV_PATH, sep=sep_bpv)
df_eda = pd.read_csv(EDA_PATH, sep=sep_eda)

df_bpv.columns = df_bpv.columns.str.strip()
df_eda.columns = df_eda.columns.str.strip()

print(f"BPV shape: {df_bpv.shape}, EDA shape: {df_eda.shape}")

# ══════════════════════════════════════════════════════════════════════════════
#  이진 분류 라벨 변환: normal(0)→0, stress(1)→1, 나머지 제거
# ══════════════════════════════════════════════════════════════════════════════
LABEL_MAP = {0: 0, 1: 1}
LABEL_NAMES = {0: 'Normal', 1: 'Stress'}

def to_binary(df, label_col='label_major'):
    df = df[df[label_col].isin(LABEL_MAP.keys())].copy()
    df[label_col] = df[label_col].map(LABEL_MAP)
    return df

df_bpv = to_binary(df_bpv)
df_eda = to_binary(df_eda)

print(f"\n[이진 분류] label_major 매핑: {LABEL_MAP}")
print(f"  BPV after filter: {df_bpv.shape}, class dist: {df_bpv['label_major'].value_counts().to_dict()}")
print(f"  EDA after filter: {df_eda.shape}, class dist: {df_eda['label_major'].value_counts().to_dict()}")

# ── TPV 피처 컬럼 식별 ──
meta_cols = ['subject', 'window', 'window_index', 'status', 'status_name',
             'label_major', 'phase', 't_start_sec', 't_end_sec', 'major_ratio',
             'subject_total_duration_sec', 'task4_duration_sec',
             'bvp_missing_ratio', 'acc_missing_ratio', 'missing_flag',
             'flatline_flag', 'clipping_ratio', 'clipping_flag', 'n_peaks',
             'peak_sparse_flag', 'ibi_pass_ratio', 'valid_beat_ratio',
             'ibi_plausibility', 'ibi_mean_sec', 'median_template_corr',
             'notch_foot_peak_detectability', 'acc_diff_mean',
             'acc_diff_exceed_ratio', 'acc_diff_flag', 'psd_corr_x',
             'psd_corr_y', 'psd_corr_z', 'psd_corr_mag', 'psd_corr_max',
             'psd_corr_flag', 'motion_score_raw', 'motion_score',
             'SQI_stress_basic', 'SQI_stress_morph', 'hard_reject',
             'motion_threshold_subject_p80', 'qc_relaxed_pass']

bpv_feat_cols = [c for c in df_bpv.columns if c not in meta_cols]
eda_feat_cols = [c for c in df_eda.columns if c not in meta_cols]

# EDA에서 사용할 선택된 TPV feature만 유지
selected_eda_tpv = [
    'TPV27', 'TPV1', 'TPV12', 'TPV16', 'TPV0', 'TPV2', 'TPV3', 'TPV4',
    'TPV9', 'TPV15', 'TPV17', 'TPV20', 'TPV21', 'TPV22', 'TPV24', 'TPV5',
    'TPV6', 'TPV8', 'TPV10', 'TPV11', 'TPV23', 'TPV28', 'TPV30', 'TPV31'
]

eda_feat_cols = [c for c in selected_eda_tpv if c in eda_feat_cols]

print(f"\nBPV feature cols ({len(bpv_feat_cols)}): {bpv_feat_cols}")
print(f"EDA feature cols ({len(eda_feat_cols)}): {eda_feat_cols}")

# ── 머지 (BPV+EDA) ──
merge_keys = ['subject', 'window_index']
df_merged = pd.merge(
    df_bpv[merge_keys + ['label_major'] + bpv_feat_cols],
    df_eda[merge_keys + eda_feat_cols],
    on=merge_keys, how='inner', suffixes=('_bpv', '_eda')
)

bpv_feat_merged = [c for c in df_merged.columns if c not in merge_keys + ['label_major']
                   and (c in bpv_feat_cols or c.endswith('_bpv'))]
eda_feat_merged = [c for c in df_merged.columns if c not in merge_keys + ['label_major']
                   and (c in eda_feat_cols or c.endswith('_eda'))
                   and c not in bpv_feat_merged]
combined_feat = bpv_feat_merged + eda_feat_merged

df_bpv_only = df_bpv[merge_keys + ['label_major'] + bpv_feat_cols].copy()
df_eda_only = df_eda[merge_keys + ['label_major'] + eda_feat_cols].copy()

print(f"\nMerged shape: {df_merged.shape}")
print(f"Merged class dist: {df_merged['label_major'].value_counts().to_dict()}")
print(f"Combined feats (before multicollinearity removal): {len(combined_feat)}")

# ══════════════════════════════════════════════════════════════════════════════
#  다중 공선성 제거 (BPV+EDA 결합 시)
# ══════════════════════════════════════════════════════════════════════════════
from statsmodels.stats.outliers_influence import variance_inflation_factor

def remove_high_corr_features(df, feat_cols, threshold=0.85):
    corr_mat = df[feat_cols].astype(float).corr(method='spearman').abs()
    label_corr = {}
    for c in feat_cols:
        vals = df[[c, 'label_major']].dropna()
        if len(vals) < 5:
            label_corr[c] = 0.0
        else:
            from scipy.stats import spearmanr
            r, _ = spearmanr(vals[c].astype(float), vals['label_major'].astype(float))
            label_corr[c] = abs(r)
    to_drop = set()
    for i in range(len(feat_cols)):
        if feat_cols[i] in to_drop:
            continue
        for j in range(i + 1, len(feat_cols)):
            if feat_cols[j] in to_drop:
                continue
            if corr_mat.iloc[i, j] > threshold:
                fi, fj = feat_cols[i], feat_cols[j]
                if label_corr[fi] >= label_corr[fj]:
                    to_drop.add(fj)
                else:
                    to_drop.add(fi)
    kept = [c for c in feat_cols if c not in to_drop]
    return kept, to_drop

def compute_vif_and_filter(df, feat_cols, vif_threshold=10.0, max_iter=50):
    remaining = list(feat_cols)
    X = df[remaining].astype(float).dropna()
    scaler = StandardScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=remaining)
    dropped_vif = []
    for iteration in range(max_iter):
        if len(remaining) <= 2:
            break
        X_cur = X_scaled[remaining].values
        vifs = []
        for k in range(len(remaining)):
            try:
                v = variance_inflation_factor(X_cur, k)
            except:
                v = np.inf
            vifs.append(v)
        max_vif = max(vifs)
        if max_vif <= vif_threshold:
            break
        worst_idx = np.argmax(vifs)
        worst_feat = remaining[worst_idx]
        dropped_vif.append((worst_feat, max_vif))
        remaining.pop(worst_idx)
    return remaining, dropped_vif

def remove_multicollinearity(df, feat_cols, corr_threshold=0.85, vif_threshold=10.0):
    print(f"\n{'='*70}")
    print(f"  다중 공선성 제거 (corr_threshold={corr_threshold}, VIF_threshold={vif_threshold})")
    print(f"{'='*70}")
    print(f"  시작 피처 수: {len(feat_cols)}")

    kept_corr, dropped_corr = remove_high_corr_features(df, feat_cols, threshold=corr_threshold)
    print(f"\n  [Step 1] Spearman |r| > {corr_threshold} 제거: {len(dropped_corr)}개 제거")
    if dropped_corr:
        for d in sorted(dropped_corr):
            print(f"    - {d}")
    print(f"  → 남은 피처: {len(kept_corr)}개")

    kept_vif, dropped_vif = compute_vif_and_filter(df, kept_corr, vif_threshold=vif_threshold)
    print(f"\n  [Step 2] VIF > {vif_threshold} 반복 제거: {len(dropped_vif)}개 제거")
    if dropped_vif:
        for feat, vif_val in dropped_vif:
            print(f"    - {feat} (VIF={vif_val:.1f})")
    print(f"  → 최종 피처: {len(kept_vif)}개")

    if len(kept_vif) >= 2:
        X_final = df[kept_vif].astype(float).dropna()
        scaler = StandardScaler()
        X_sc = scaler.fit_transform(X_final)
        final_vifs = []
        for k in range(len(kept_vif)):
            try:
                v = variance_inflation_factor(X_sc, k)
            except:
                v = np.inf
            final_vifs.append(v)
        print(f"\n  최종 VIF 확인:")
        for feat, vif_val in zip(kept_vif, final_vifs):
            print(f"    {feat:30s} VIF={vif_val:.2f}")

    total_removed = len(feat_cols) - len(kept_vif)
    print(f"\n  총 제거: {total_removed}개 ({len(feat_cols)} → {len(kept_vif)})")
    return kept_vif

combined_feat_clean = remove_multicollinearity(
    df_merged, combined_feat, corr_threshold=0.85, vif_threshold=10.0
)

# ══════════════════════════════════════════════════════════════════════════════
#  상관관계 분석
# ══════════════════════════════════════════════════════════════════════════════
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pointbiserialr, spearmanr

OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/outputs_empatica_e4stress"
os.makedirs(OUT_DIR, exist_ok=True)

def plot_feature_label_corr(df, feat_cols, label_col, title, fname):
    corrs = []
    for c in feat_cols:
        vals = df[[c, label_col]].dropna()
        if len(vals) < 5:
            continue
        x, y = vals[c].astype(float), vals[label_col].astype(float)
        try:
            pb_r, pb_p = pointbiserialr(y, x)
        except:
            pb_r, pb_p = np.nan, np.nan
        sp_r, sp_p = spearmanr(x, y)
        corrs.append({'Feature': c, 'PointBiserial_r': pb_r, 'PB_pval': pb_p,
                       'Spearman_r': sp_r, 'SP_pval': sp_p})
    df_corr = pd.DataFrame(corrs).sort_values('Spearman_r', key=abs, ascending=False)

    print(f"\n{'─'*70}")
    print(f"  {title}: Feature-Label Correlation (top 15) [Binary: Normal vs Stress]")
    print(f"{'─'*70}")
    print(df_corr.head(15).to_string(index=False))

    top = df_corr.head(20)
    fig, ax = plt.subplots(figsize=(10, max(4, len(top)*0.35)))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top['Spearman_r']]
    ax.barh(range(len(top)), top['Spearman_r'].values, color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Feature'].values, fontsize=8)
    ax.set_xlabel('Spearman r')
    ax.set_title(f'{title} — Feature-Label Spearman Correlation (Binary)')
    ax.axvline(0, color='k', linewidth=0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_corr

def plot_feature_corr_matrix(df, feat_cols, title, fname):
    corr_mat = df[feat_cols].astype(float).corr(method='spearman')
    fig, ax = plt.subplots(figsize=(max(8, len(feat_cols)*0.5),
                                     max(6, len(feat_cols)*0.4)))
    mask = np.triu(np.ones_like(corr_mat, dtype=bool), k=1)
    sns.heatmap(corr_mat, mask=mask, cmap='RdBu_r', center=0,
                vmin=-1, vmax=1, annot=len(feat_cols) <= 20,
                fmt='.2f' if len(feat_cols) <= 20 else '',
                square=True, linewidths=0.5,
                cbar_kws={'shrink': 0.8}, ax=ax)
    ax.set_title(f'{title} — Feature Correlation Matrix (Spearman)', fontsize=11)
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    pairs = []
    for i in range(len(feat_cols)):
        for j in range(i+1, len(feat_cols)):
            r = corr_mat.iloc[i, j]
            if abs(r) > 0.7:
                pairs.append((feat_cols[i], feat_cols[j], r))
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    if pairs:
        print(f"\n  {title}: 높은 피처간 상관 (|r|>0.7) — {len(pairs)}쌍")
        for a, b, r in pairs[:15]:
            print(f"    {a:20s} ↔ {b:20s} : r={r:.4f}")
    return corr_mat

def plot_cross_corr_bpv_eda(df_merged, bpv_feats, eda_feats, fname):
    all_feats = bpv_feats + eda_feats
    corr_full = df_merged[all_feats].astype(float).corr(method='spearman')
    cross = corr_full.loc[bpv_feats, eda_feats]
    fig, ax = plt.subplots(figsize=(max(8, len(eda_feats)*0.5),
                                     max(5, len(bpv_feats)*0.4)))
    sns.heatmap(cross, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=max(len(bpv_feats), len(eda_feats)) <= 20,
                fmt='.2f' if max(len(bpv_feats), len(eda_feats)) <= 20 else '',
                linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
    ax.set_title('BPV vs EDA — Cross-Feature Correlation (Spearman, Binary)', fontsize=11)
    ax.set_ylabel('BPV Features')
    ax.set_xlabel('EDA Features')
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    print(f"\n{'─'*70}")
    print(f"  BPV ↔ EDA Cross-Correlation (top 15 by |r|)")
    print(f"{'─'*70}")
    cross_pairs = []
    for b in bpv_feats:
        for e in eda_feats:
            cross_pairs.append((b, e, cross.loc[b, e]))
    cross_pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    for a, b, r in cross_pairs[:15]:
        print(f"    {a:20s} ↔ {b:20s} : r={r:.4f}")
    return cross

def plot_subject_corr_comparison(df_src, feat_cols, label_col, title, fname):
    subjects = sorted(df_src['subject'].unique())
    corr_dict = {}
    for subj in subjects:
        ds = df_src[df_src['subject'] == subj]
        rs = []
        for c in feat_cols:
            vals = ds[[c, label_col]].dropna()
            if len(vals) < 5:
                rs.append(np.nan)
            else:
                r, _ = spearmanr(vals[c].astype(float), vals[label_col].astype(float))
                rs.append(r)
        corr_dict[subj] = rs
    df_subj_corr = pd.DataFrame(corr_dict, index=feat_cols).T
    fig, ax = plt.subplots(figsize=(max(8, len(feat_cols)*0.5),
                                     max(4, len(subjects)*0.4)))
    sns.heatmap(df_subj_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
                annot=len(feat_cols) <= 15 and len(subjects) <= 20,
                fmt='.2f' if len(feat_cols) <= 15 else '',
                linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax)
    ax.set_title(f'{title} — Subject-wise Feature-Label Correlation (Binary)', fontsize=11)
    ax.set_ylabel('Subject')
    plt.xticks(fontsize=7, rotation=45, ha='right')
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_subj_corr

# ── 실행: 상관분석 ──
print(f"\n\n{'='*80}")
print("  상관관계 분석 (Binary: Normal vs Stress)")
print(f"{'='*80}")

corr_bpv_label = plot_feature_label_corr(
    df_bpv_only, bpv_feat_cols, 'label_major', 'BPV', 'corr_bpv_label.png')
corr_eda_label = plot_feature_label_corr(
    df_eda_only, eda_feat_cols, 'label_major', 'EDA', 'corr_eda_label.png')
corr_comb_label_raw = plot_feature_label_corr(
    df_merged, combined_feat, 'label_major', 'BPV+EDA (raw)', 'corr_combined_label_raw.png')
corr_comb_label = plot_feature_label_corr(
    df_merged, combined_feat_clean, 'label_major', 'BPV+EDA (clean)', 'corr_combined_label.png')

plot_feature_corr_matrix(df_bpv_only, bpv_feat_cols, 'BPV', 'corr_matrix_bpv.png')
plot_feature_corr_matrix(df_eda_only, eda_feat_cols, 'EDA', 'corr_matrix_eda.png')
plot_feature_corr_matrix(df_merged, combined_feat_clean, 'BPV+EDA (clean)', 'corr_matrix_combined_clean.png')
plot_cross_corr_bpv_eda(df_merged, bpv_feat_merged, eda_feat_merged, 'corr_cross_bpv_eda.png')

subj_corr_bpv = plot_subject_corr_comparison(
    df_bpv_only, bpv_feat_cols, 'label_major', 'BPV', 'corr_subject_bpv.png')
subj_corr_eda = plot_subject_corr_comparison(
    df_eda_only, eda_feat_cols, 'label_major', 'EDA', 'corr_subject_eda.png')

corr_bpv_label.to_csv(os.path.join(OUT_DIR, 'corr_bpv_label.csv'), index=False)
corr_eda_label.to_csv(os.path.join(OUT_DIR, 'corr_eda_label.csv'), index=False)
corr_comb_label.to_csv(os.path.join(OUT_DIR, 'corr_combined_label.csv'), index=False)

print(f"\n상관분석 그래프/CSV 저장 완료 → {OUT_DIR}/")

# ══════════════════════════════════════════════════════════════════════════════
#  분류기 정의
# ══════════════════════════════════════════════════════════════════════════════
def get_classifiers():
    return {
        'RF': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'DT': DecisionTreeClassifier(random_state=42),
        'SVC': SVC(kernel='rbf', probability=True, random_state=42),
        'LR': LogisticRegression(max_iter=2000, random_state=42)
    }

# ══════════════════════════════════════════════════════════════════════════════
#  LOSO (Leave-One-Subject-Out) 이진 분류
# ══════════════════════════════════════════════════════════════════════════════
feature_configs = {
    'BPV': (df_bpv_only, bpv_feat_cols),
    'EDA': (df_eda_only, eda_feat_cols),
    'BPV+EDA': (df_merged, combined_feat_clean),
}

all_results = []

subjects = sorted(df_merged['subject'].unique())
print(f"\n{'='*80}")
print(f"  LOSO Cross-Subject Binary Classification (Normal vs Stress)")
print(f"  총 {len(subjects)} 명 피험자: {subjects}")
print(f"{'='*80}\n")

for feat_name, (df_src, feat_cols) in feature_configs.items():
    print(f"\n{'#'*80}")
    print(f"  Feature Set: {feat_name} ({len(feat_cols)} features) — LOSO")
    print(f"{'#'*80}")

    for test_subj in subjects:
        df_train = df_src[df_src['subject'] != test_subj].copy()
        df_test  = df_src[df_src['subject'] == test_subj].copy()

        df_train = df_train.dropna(subset=feat_cols + ['label_major'])
        df_test  = df_test.dropna(subset=feat_cols + ['label_major'])

        X_train = df_train[feat_cols].values.astype(float)
        y_train = df_train['label_major'].values.astype(int)
        X_test  = df_test[feat_cols].values.astype(float)
        y_test  = df_test['label_major'].values.astype(int)

        n_train_cls = len(np.unique(y_train))
        n_test_cls  = len(np.unique(y_test))

        if len(df_test) < 2 or n_train_cls < 2:
            print(f"\n  [Test={test_subj}] 데이터 부족 → SKIP")
            continue

        train_dist = dict(zip(*np.unique(y_train, return_counts=True)))
        test_dist  = dict(zip(*np.unique(y_test, return_counts=True)))

        print(f"\n  ── Test Subject: {test_subj} | "
              f"Train={len(X_train)} ({train_dist}), "
              f"Test={len(X_test)} ({test_dist}) ──")

        scaler = StandardScaler()
        X_tr_sc = scaler.fit_transform(X_train)
        X_te_sc = scaler.transform(X_test)

        for clf_name, clf in get_classifiers().items():
            clf.fit(X_tr_sc, y_train)
            y_pred = clf.predict(X_te_sc)

            acc = accuracy_score(y_test, y_pred)
            f1  = f1_score(y_test, y_pred, average='binary', zero_division=0)

            try:
                y_prob = clf.predict_proba(X_te_sc)[:, 1]
                auc = roc_auc_score(y_test, y_prob) if n_test_cls == 2 else np.nan
            except:
                auc = np.nan

            cm = confusion_matrix(y_test, y_pred, labels=[0, 1])

            all_results.append({
                'Feature': feat_name,
                'Test_Subject': test_subj,
                'Classifier': clf_name,
                'Accuracy': acc,
                'F1_binary': f1,
                'AUC': auc,
                'TN': cm[0, 0], 'FP': cm[0, 1],
                'FN': cm[1, 0], 'TP': cm[1, 1],
                'N_train': len(X_train),
                'N_test': len(X_test),
            })

            print(f"    {clf_name:4s} | Acc={acc:.4f} | F1={f1:.4f} | "
                  f"AUC={auc:.4f} | CM=[TN={cm[0,0]} FP={cm[0,1]} FN={cm[1,0]} TP={cm[1,1]}]")

# ══════════════════════════════════════════════════════════════════════════════
#  결과 요약
# ══════════════════════════════════════════════════════════════════════════════
df_res = pd.DataFrame(all_results)

print(f"\n\n{'='*80}")
print("  LOSO 결과 요약 — Binary (Normal vs Stress)")
print(f"{'='*80}")

for feat_name in ['BPV', 'EDA', 'BPV+EDA']:
    sub = df_res[df_res['Feature'] == feat_name]
    if sub.empty:
        continue
    print(f"\n── {feat_name} ──")

    pivot_acc = sub.pivot_table(index='Test_Subject', columns='Classifier',
                                values='Accuracy', aggfunc='first')
    pivot_f1 = sub.pivot_table(index='Test_Subject', columns='Classifier',
                               values='F1_binary', aggfunc='first')
    pivot_auc = sub.pivot_table(index='Test_Subject', columns='Classifier',
                                values='AUC', aggfunc='first')

    print("\n[Accuracy]")
    print(pivot_acc.round(4).to_string())
    print(f"  Mean: {pivot_acc.mean().round(4).to_dict()}")
    print(f"  Std:  {pivot_acc.std().round(4).to_dict()}")

    print("\n[F1 binary]")
    print(pivot_f1.round(4).to_string())
    print(f"  Mean: {pivot_f1.mean().round(4).to_dict()}")
    print(f"  Std:  {pivot_f1.std().round(4).to_dict()}")

    print("\n[AUC]")
    print(pivot_auc.round(4).to_string())
    print(f"  Mean: {pivot_auc.mean().round(4).to_dict()}")
    print(f"  Std:  {pivot_auc.std().round(4).to_dict()}")

# ── Feature set 간 평균 비교 ──
print(f"\n\n{'='*80}")
print("  Feature Set × Classifier 평균 성능 비교 (LOSO)")
print(f"{'='*80}")
summary_mean = df_res.groupby(['Feature', 'Classifier'])[['Accuracy', 'F1_binary', 'AUC']].mean()
summary_std  = df_res.groupby(['Feature', 'Classifier'])[['Accuracy', 'F1_binary', 'AUC']].std()

summary_combined = summary_mean.round(4).astype(str) + ' ± ' + summary_std.round(4).astype(str)
print(summary_combined.to_string())

# ── CSV 저장 ──
OUT_PATH = "/content/drive/MyDrive/Colab Notebooks/loso_binary_results_empatica_e4stress.csv"
df_res.to_csv(OUT_PATH, index=False)
print(f"\n결과 저장: {OUT_PATH}")

# ── 제거된 피처 정보 저장 ──
removed_feats = [c for c in combined_feat if c not in combined_feat_clean]
df_removed = pd.DataFrame({
    'removed_feature': removed_feats,
    'source': ['BPV' if c in bpv_feat_merged else 'EDA' for c in removed_feats]
})
df_removed.to_csv(os.path.join(OUT_DIR, 'removed_multicollinear_features.csv'), index=False)
print(f"\n제거된 피처 수: {len(removed_feats)}, 최종 BPV+EDA 피처 수: {len(combined_feat_clean)}")

BPV shape: (1035, 53), EDA shape: (516, 74)

[이진 분류] label_major 매핑: {0: 0, 1: 1}
  BPV after filter: (1035, 53), class dist: {1: 671, 0: 364}
  EDA after filter: (516, 74), class dist: {1: 334, 0: 182}

BPV feature cols (11): ['TPV22', 'TPV9', 'TPV4', 'TPV2', 'TPV0', 'TPV28', 'TPV8', 'TPV19', 'TPV14', 'TPV26', 'TPV11']
EDA feature cols (24): ['TPV27', 'TPV1', 'TPV12', 'TPV16', 'TPV0', 'TPV2', 'TPV3', 'TPV4', 'TPV9', 'TPV15', 'TPV17', 'TPV20', 'TPV21', 'TPV22', 'TPV24', 'TPV5', 'TPV6', 'TPV8', 'TPV10', 'TPV11', 'TPV23', 'TPV28', 'TPV30', 'TPV31']

Merged shape: (516, 38)
Merged class dist: {1: 338, 0: 178}
Combined feats (before multicollinearity removal): 35

  다중 공선성 제거 (corr_threshold=0.85, VIF_threshold=10.0)
  시작 피처 수: 35

  [Step 1] Spearman |r| > 0.85 제거: 19개 제거
    - TPV0_bpv
    - TPV1
    - TPV11_eda
    - TPV12
    - TPV14
    - TPV15
    - TPV16
    - TPV17
    - TPV19
    - TPV20
    - TPV21
    - TPV23
    - TPV26
    - TPV28_eda
    - TPV2_eda
    - TPV30
    - TPV31
   

## 3) TPV_22 only Personalized

In [5]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    confusion_matrix
)
from sklearn.preprocessing import StandardScaler

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pointbiserialr, spearmanr

import warnings, os
warnings.filterwarnings('ignore')

# ── 파일 경로 ──
BPV_PATH = "/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP_TPV/noQC_BVP.csv"

# ── 데이터 로드 ──
sep_bpv = '\t' if open(BPV_PATH).readline().count('\t') > 3 else ','
df_bpv = pd.read_csv(BPV_PATH, sep=sep_bpv)
df_bpv.columns = df_bpv.columns.str.strip()

print(f"BPV shape: {df_bpv.shape}")
print(f"BPV columns: {list(df_bpv.columns)}")

# ══════════════════════════════════════════════════════════════════════════════
# EmpaticaE4Stress 이진 분류 라벨 처리
# label_major: normal(0), stress(1)
# ══════════════════════════════════════════════════════════════════════════════
LABEL_MAP = {0: 0, 1: 1}
LABEL_NAMES = {0: 'Normal', 1: 'Stress'}

def to_binary(df, label_col='label_major'):
    df = df[df[label_col].isin(LABEL_MAP.keys())].copy()
    df[label_col] = df[label_col].map(LABEL_MAP)
    return df

df_bpv = to_binary(df_bpv)

print(f"\n[이진 분류] label_major 매핑: {LABEL_MAP}")
print(f"  BPV after filter: {df_bpv.shape}, class dist: {df_bpv['label_major'].value_counts().to_dict()}")

# ── TPV 피처 컬럼 식별 ──
meta_cols = [
    'subject', 'window', 'window_index', 'status', 'status_name',
    'label_major', 'phase', 't_start_sec', 't_end_sec', 'major_ratio',
    'subject_total_duration_sec', 'task4_duration_sec',
    'bvp_missing_ratio', 'acc_missing_ratio', 'missing_flag',
    'flatline_flag', 'clipping_ratio', 'clipping_flag', 'n_peaks',
    'peak_sparse_flag', 'ibi_pass_ratio', 'valid_beat_ratio',
    'ibi_plausibility', 'ibi_mean_sec', 'median_template_corr',
    'notch_foot_peak_detectability', 'acc_diff_mean',
    'acc_diff_exceed_ratio', 'acc_diff_flag', 'psd_corr_x',
    'psd_corr_y', 'psd_corr_z', 'psd_corr_mag', 'psd_corr_max',
    'psd_corr_flag', 'motion_score_raw', 'motion_score',
    'SQI_stress_basic', 'SQI_stress_morph', 'hard_reject',
    'motion_threshold_subject_p80', 'qc_relaxed_pass'
]

bpv_feat_cols = [c for c in df_bpv.columns if c not in meta_cols]

print(f"\nBPV feature cols ({len(bpv_feat_cols)}): {bpv_feat_cols}")

# ══════════════════════════════════════════════════════════════════════════════
# TPV_22만 사용
# ══════════════════════════════════════════════════════════════════════════════
TARGET_FEAT = 'TPV22'

if TARGET_FEAT not in df_bpv.columns:
    raise ValueError(f"{TARGET_FEAT} 컬럼이 BPV 파일에 없습니다.")

merge_keys = ['subject', 'window_index']
df_bpv_only = df_bpv[merge_keys + ['label_major', TARGET_FEAT]].copy()

print(f"\n사용 피처: {TARGET_FEAT}")

# ══════════════════════════════════════════════════════════════════════════════
# 상관관계 분석
# ══════════════════════════════════════════════════════════════════════════════
OUT_DIR = "/content/drive/MyDrive/Colab Notebooks/outputs_empatica_e4stress_tpv22_lr"
os.makedirs(OUT_DIR, exist_ok=True)

def plot_feature_label_corr(df, feat_cols, label_col, title, fname):
    corrs = []
    for c in feat_cols:
        vals = df[[c, label_col]].dropna()
        if len(vals) < 5:
            continue
        x, y = vals[c].astype(float), vals[label_col].astype(float)
        try:
            pb_r, pb_p = pointbiserialr(y, x)
        except:
            pb_r, pb_p = np.nan, np.nan
        sp_r, sp_p = spearmanr(x, y)
        corrs.append({
            'Feature': c,
            'PointBiserial_r': pb_r,
            'PB_pval': pb_p,
            'Spearman_r': sp_r,
            'SP_pval': sp_p
        })

    df_corr = pd.DataFrame(corrs).sort_values('Spearman_r', key=abs, ascending=False)

    print(f"\n{'─'*70}")
    print(f"  {title}: Feature-Label Correlation [Binary: Normal vs Stress]")
    print(f"{'─'*70}")
    print(df_corr.to_string(index=False))

    top = df_corr.head(20)
    fig, ax = plt.subplots(figsize=(8, max(4, len(top) * 0.5)))
    colors = ['#e74c3c' if v < 0 else '#2ecc71' for v in top['Spearman_r']]
    ax.barh(range(len(top)), top['Spearman_r'].values, color=colors)
    ax.set_yticks(range(len(top)))
    ax.set_yticklabels(top['Feature'].values, fontsize=9)
    ax.set_xlabel('Spearman r')
    ax.set_title(f'{title} — Feature-Label Spearman Correlation')
    ax.axvline(0, color='k', linewidth=0.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_corr

def plot_subject_corr_comparison(df_src, feat_cols, label_col, title, fname):
    subjects = sorted(df_src['subject'].unique())
    corr_dict = {}
    for subj in subjects:
        ds = df_src[df_src['subject'] == subj]
        rs = []
        for c in feat_cols:
            vals = ds[[c, label_col]].dropna()
            if len(vals) < 5:
                rs.append(np.nan)
            else:
                r, _ = spearmanr(vals[c].astype(float), vals[label_col].astype(float))
                rs.append(r)
        corr_dict[subj] = rs

    df_subj_corr = pd.DataFrame(corr_dict, index=feat_cols).T

    fig, ax = plt.subplots(figsize=(6, max(4, len(subjects) * 0.4)))
    sns.heatmap(
        df_subj_corr, cmap='RdBu_r', center=0, vmin=-1, vmax=1,
        annot=True, fmt='.2f', linewidths=0.5, cbar_kws={'shrink': 0.8}, ax=ax
    )
    ax.set_title(f'{title} — Subject-wise Feature-Label Correlation', fontsize=11)
    ax.set_ylabel('Subject')
    plt.xticks(fontsize=8, rotation=0)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, fname), dpi=150)
    plt.close()
    return df_subj_corr

print(f"\n\n{'='*80}")
print("  상관관계 분석 (Binary: Normal vs Stress)")
print(f"{'='*80}")

corr_bpv_label = plot_feature_label_corr(
    df_bpv_only, [TARGET_FEAT], 'label_major', 'BPV-TPV_22', 'corr_bpv_tpv22_label.png'
)

subj_corr_bpv = plot_subject_corr_comparison(
    df_bpv_only, [TARGET_FEAT], 'label_major', 'BPV-TPV_22', 'corr_subject_bpv_tpv22.png'
)

corr_bpv_label.to_csv(os.path.join(OUT_DIR, 'corr_bpv_tpv22_label.csv'), index=False)
print(f"\n상관분석 그래프 저장 완료 → {OUT_DIR}/")
print(f"상관계수 CSV 저장 완료 → {OUT_DIR}/corr_bpv_tpv22_label.csv")

# ── 분류기 정의: LR만 사용 ──
def get_classifier():
    return LogisticRegression(max_iter=2000, random_state=42)

# ── 평가 함수 ──
def evaluate(clf, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_tr = scaler.fit_transform(X_train)
    X_te = scaler.transform(X_test)

    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_te)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=0)

    try:
        y_prob = clf.predict_proba(X_te)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    except:
        auc = np.nan

    cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
    return acc, f1, auc, cm, y_pred

# ── 실험 실행: subject-wise personalized split ──
all_results = []

subjects = sorted(df_bpv_only['subject'].unique())
print(f"\n{'='*80}")
print(f"총 {len(subjects)} 명 피험자: {subjects}")
print(f"이진 분류: Normal(0) vs Stress(1)")
print(f"Feature: BPV-TPV_22 | Classifier: LR")
print(f"{'='*80}\n")

for subj in subjects:
    df_subj = df_bpv_only[df_bpv_only['subject'] == subj].copy()
    df_subj = df_subj.dropna(subset=[TARGET_FEAT, 'label_major'])

    X = df_subj[[TARGET_FEAT]].values.astype(float)
    y = df_subj['label_major'].values.astype(int)

    n_classes = len(np.unique(y))
    if len(df_subj) < 5 or n_classes < 2:
        print(f"\n  [{subj}] 샘플 부족 (n={len(df_subj)}, classes={n_classes}) → SKIP")
        continue

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=42, stratify=y
    )

    print(f"\n  ── {subj} | Train={len(X_train)}, Test={len(X_test)}, "
          f"Class dist: {dict(zip(*np.unique(y, return_counts=True)))} ──")

    clf = get_classifier()
    acc, f1, auc, cm, y_pred = evaluate(clf, X_train, X_test, y_train, y_test)

    all_results.append({
        'Feature': 'BPV-TPV_22',
        'Subject': subj,
        'Classifier': 'LR',
        'Accuracy': acc,
        'F1_binary': f1,
        'AUC': auc,
    })

    print(f"    LR   | Acc={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | CM={cm.tolist()}")

# ── 결과 요약 ──
df_res = pd.DataFrame(all_results)

print(f"\n\n{'='*80}")
print("  전체 결과 요약 — Binary Classification (Normal vs Stress)")
print(f"{'='*80}")

if not df_res.empty:
    result_table = df_res[['Subject', 'Accuracy', 'F1_binary', 'AUC']].copy()
    result_table = result_table.sort_values('Subject')

    print("\n[Subject-wise Results]")
    print(result_table.round(4).to_string(index=False))

    print("\n[Average]")
    print(result_table[['Accuracy', 'F1_binary', 'AUC']].mean().round(4).to_dict())

# ── CSV 저장 ──
OUT_PATH = "/content/drive/MyDrive/Colab Notebooks/personalized_results_binary_empatica_e4stress_BPV_TPV22_LR.csv"
df_res.to_csv(OUT_PATH, index=False)
print(f"\n결과 저장: {OUT_PATH}")

BPV shape: (1035, 53)
BPV columns: ['subject', 'window', 'window_index', 'status', 'status_name', 'label_major', 'phase', 't_start_sec', 't_end_sec', 'major_ratio', 'subject_total_duration_sec', 'task4_duration_sec', 'bvp_missing_ratio', 'acc_missing_ratio', 'missing_flag', 'flatline_flag', 'clipping_ratio', 'clipping_flag', 'n_peaks', 'peak_sparse_flag', 'ibi_pass_ratio', 'valid_beat_ratio', 'ibi_plausibility', 'ibi_mean_sec', 'median_template_corr', 'notch_foot_peak_detectability', 'acc_diff_mean', 'acc_diff_exceed_ratio', 'acc_diff_flag', 'psd_corr_x', 'psd_corr_y', 'psd_corr_z', 'psd_corr_mag', 'psd_corr_max', 'psd_corr_flag', 'motion_score_raw', 'motion_score', 'SQI_stress_basic', 'SQI_stress_morph', 'hard_reject', 'motion_threshold_subject_p80', 'qc_relaxed_pass', 'TPV22', 'TPV9', 'TPV4', 'TPV2', 'TPV0', 'TPV28', 'TPV8', 'TPV19', 'TPV14', 'TPV26', 'TPV11']

[이진 분류] label_major 매핑: {0: 0, 1: 1}
  BPV after filter: (1035, 53), class dist: {1: 671, 0: 364}

BPV feature cols (11): ['

## 4) Noise Robustness

In [8]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import f_oneway, kruskal

warnings.filterwarnings("ignore")

# =============================================================================
# CONFIG
# =============================================================================
OUTPUT_DIR = r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/noise_robustness_TPV22_RMSSD"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# 이미 추출된 CSV 경로만 설정하면 됨
# noise key는 아래 4개를 그대로 유지
#   orig, alpha_0.01, alpha_0.03, alpha_0.05
# -----------------------------------------------------------------------------
TPV_CSV_PATHS = {
    "orig":       r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP_TPV/noQC_BVP.csv",
    "alpha_0.01": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP-TPV-noise/noQC_BVP_alpha_0.01.csv",
    "alpha_0.03": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP-TPV-noise/noQC_BVP_alpha_0.03.csv",
    "alpha_0.05": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP-TPV-noise/noQC_BVP_alpha_0.05.csv",
}

HRV_CSV_PATHS = {
    "orig":       r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV/EmpaticaE4Stress_IBI_HRV_1min.csv",
    "alpha_0.01": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV-noise/EmpaticaE4Stress_BVP_HRV_1min_alpha_0.01.csv",
    "alpha_0.03": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV-noise/EmpaticaE4Stress_BVP_HRV_1min_alpha_0.03.csv",
    "alpha_0.05": r"/content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV-noise/EmpaticaE4Stress_BVP_HRV_1min_alpha_0.05.csv",
}

NOISE_LEVELS = {
    "orig": 0.00,
    "alpha_0.01": 0.01,
    "alpha_0.03": 0.03,
    "alpha_0.05": 0.05,
}

# EmpaticaE4Stress는 보통 normal/stress. 없으면 자동으로 안나옴 -> 확장성 고려
LABEL_TO_STATUS = {
    0: "normal",
    1: "stress",
    2: "baseline",
    4: "meditation",
}

STATUS_ORDER = ["normal", "stress", "baseline", "meditation"]
NOISE_ORDER = ["orig", "alpha_0.01", "alpha_0.03", "alpha_0.05"]


# =============================================================================
# UTILS
# =============================================================================
def detect_sep(csv_path: str):
    with open(csv_path, "r", encoding="utf-8") as f:
        line = f.readline()
    return "\t" if line.count("\t") > 3 else ","


def normalize_status_name(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().lower()
    mapping = {
        "normal": "normal",
        "stress": "stress",
        "baseline": "baseline",
        "meditation": "meditation",
        "amusement": "meditation",
        "rest": "normal",
    }
    return mapping.get(s, s)


def ensure_status_name(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    if "status_name" in df.columns:
        df["status_name"] = df["status_name"].apply(normalize_status_name)
        return df

    if "label_major" in df.columns:
        df["status_name"] = df["label_major"].map(LABEL_TO_STATUS)
        df["status_name"] = df["status_name"].apply(normalize_status_name)
        return df

    if "status" in df.columns:
        df["status_name"] = df["status"].apply(normalize_status_name)
        return df

    raise ValueError("status_name / label_major / status 컬럼이 없습니다.")


def find_tpv22_col(df: pd.DataFrame):
    candidates = ["TPV22", "TPV_22", "tpv22", "tpv_22"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("TPV22 또는 TPV_22 컬럼을 찾을 수 없습니다.")


def find_rmssd_col(df: pd.DataFrame):
    candidates = ["RMSSD", "rmssd"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError("RMSSD 컬럼을 찾을 수 없습니다.")


def get_common_merge_cols(df1: pd.DataFrame, df2: pd.DataFrame):
    preferred = [
        "subject", "window", "window_index", "status", "status_name",
        "label_major", "phase", "t_start_sec", "t_end_sec", "major_ratio"
    ]
    common = [c for c in preferred if c in df1.columns and c in df2.columns]
    if len(common) == 0:
        raise ValueError("TPV와 HRV CSV 사이에 공통 merge column이 없습니다.")
    return common


# =============================================================================
# LOAD + MERGE PRE-EXTRACTED CSVs
# =============================================================================
def load_one_noise_pair(noise_name: str, tpv_path: str, hrv_path: str) -> pd.DataFrame:
    print(f"\n[INFO] Loading noise={noise_name}")
    print(f"       TPV: {tpv_path}")
    print(f"       HRV: {hrv_path}")

    if not os.path.exists(tpv_path):
        raise FileNotFoundError(f"TPV CSV not found: {tpv_path}")
    if not os.path.exists(hrv_path):
        raise FileNotFoundError(f"HRV CSV not found: {hrv_path}")

    df_tpv = pd.read_csv(tpv_path, sep=detect_sep(tpv_path))
    df_hrv = pd.read_csv(hrv_path, sep=detect_sep(hrv_path))

    df_tpv.columns = df_tpv.columns.str.strip()
    df_hrv.columns = df_hrv.columns.str.strip()

    df_tpv = ensure_status_name(df_tpv)
    df_hrv = ensure_status_name(df_hrv)

    tpv22_col = find_tpv22_col(df_tpv)
    rmssd_col = find_rmssd_col(df_hrv)

    merge_cols = get_common_merge_cols(df_tpv, df_hrv)

    keep_tpv = merge_cols + [tpv22_col]
    keep_hrv = merge_cols + [rmssd_col]

    df_tpv = df_tpv[keep_tpv].copy()
    df_hrv = df_hrv[keep_hrv].copy()

    df_tpv = df_tpv.rename(columns={tpv22_col: "TPV22"})
    df_hrv = df_hrv.rename(columns={rmssd_col: "RMSSD"})

    df_merged = pd.merge(df_tpv, df_hrv, on=merge_cols, how="inner")

    df_merged["noise"] = noise_name
    df_merged["alpha"] = NOISE_LEVELS[noise_name]

    print(f"[INFO] merged shape ({noise_name}): {df_merged.shape}")
    print(f"[INFO] status dist ({noise_name}): {df_merged['status_name'].value_counts(dropna=False).to_dict()}")

    return df_merged


def load_all_preextracted_csvs():
    dfs = []

    for noise_name in NOISE_ORDER:
        df_one = load_one_noise_pair(
            noise_name=noise_name,
            tpv_path=TPV_CSV_PATHS[noise_name],
            hrv_path=HRV_CSV_PATHS[noise_name],
        )
        dfs.append(df_one)

    df_all = pd.concat(dfs, axis=0, ignore_index=True)
    return df_all


# =============================================================================
# STATS
# =============================================================================
def eta_squared_anova(groups):
    all_vals = np.concatenate(groups)
    grand_mean = np.mean(all_vals)

    ss_between = 0.0
    ss_total = np.sum((all_vals - grand_mean) ** 2)

    for g in groups:
        if len(g) == 0:
            continue
        ss_between += len(g) * (np.mean(g) - grand_mean) ** 2

    if ss_total == 0:
        return np.nan
    return ss_between / ss_total


def consistency_ratio(df, value_col, label_col="status_name"):
    grouped = df.groupby(label_col)[value_col]
    means = grouped.mean()
    stds = grouped.std()

    if len(means) < 2:
        return np.nan

    denom = stds.mean()
    if pd.isna(denom) or denom == 0:
        return np.nan

    return means.std() / denom


def get_present_status_order(df: pd.DataFrame):
    present = [s for s in STATUS_ORDER if s in set(df["status_name"].dropna().unique())]
    return present


def summarize_by_noise_and_feature(df, value_col):
    results = []
    order_noise = NOISE_ORDER

    for noise_name in order_noise:
        sub = df[df["noise"] == noise_name].copy()
        sub = sub[np.isfinite(sub[value_col])].copy()

        label_order = get_present_status_order(sub)

        groups = []
        label_used = []
        means = {}
        stds = {}
        ns = {}

        for lb in label_order:
            vals = sub.loc[sub["status_name"] == lb, value_col].dropna().values
            if len(vals) > 0:
                groups.append(vals)
                label_used.append(lb)
                means[lb] = float(np.mean(vals))
                stds[lb] = float(np.std(vals, ddof=0))
                ns[lb] = int(len(vals))

        alpha_val = NOISE_LEVELS[noise_name]

        row = {
            "feature": value_col,
            "noise": noise_name,
            "alpha": alpha_val,
            "n_groups": len(groups),
            "labels_used": ",".join(label_used),
            "anova_F": np.nan,
            "anova_p": np.nan,
            "kruskal_H": np.nan,
            "kruskal_p": np.nan,
            "eta_sq": np.nan,
            "consistency_ratio": np.nan,

            "normal_n": ns.get("normal", 0),
            "stress_n": ns.get("stress", 0),
            "baseline_n": ns.get("baseline", 0),
            "meditation_n": ns.get("meditation", 0),

            "normal_mean": means.get("normal", np.nan),
            "stress_mean": means.get("stress", np.nan),
            "baseline_mean": means.get("baseline", np.nan),
            "meditation_mean": means.get("meditation", np.nan),

            "normal_std": stds.get("normal", np.nan),
            "stress_std": stds.get("stress", np.nan),
            "baseline_std": stds.get("baseline", np.nan),
            "meditation_std": stds.get("meditation", np.nan),
        }

        if len(groups) >= 2:
            try:
                f_stat, p_val = f_oneway(*groups)
                row["anova_F"] = float(f_stat)
                row["anova_p"] = float(p_val)
            except Exception:
                pass

            try:
                h_stat, p_kw = kruskal(*groups)
                row["kruskal_H"] = float(h_stat)
                row["kruskal_p"] = float(p_kw)
            except Exception:
                pass

            try:
                row["eta_sq"] = float(eta_squared_anova(groups))
            except Exception:
                pass

            try:
                row["consistency_ratio"] = float(consistency_ratio(sub, value_col))
            except Exception:
                pass

        results.append(row)

    return pd.DataFrame(results)


def build_table12_style(stats_df):
    rows = []

    for feature in ["TPV22", "RMSSD"]:
        sub = stats_df[stats_df["feature"] == feature].copy()

        for noise_name in NOISE_ORDER:
            one = sub[sub["noise"] == noise_name]
            if len(one) == 0:
                continue
            r = one.iloc[0]

            for cond in ["normal", "stress", "baseline", "meditation"]:
                mean_col = f"{cond}_mean"
                std_col = f"{cond}_std"
                n_col = f"{cond}_n"

                if n_col in r and r[n_col] > 0:
                    rows.append({
                        "Feature": feature,
                        "Noise Level": "Clean" if noise_name == "orig" else f"{r['alpha']:.2f}",
                        "Condition": cond.capitalize(),
                        "Mean ± STD": f"{r[mean_col]:.3f} ± {r[std_col]:.3f}",
                        "p-value": "< 0.001" if pd.notna(r["anova_p"]) and r["anova_p"] < 0.001 else (
                            f"{r['anova_p']:.3f}" if pd.notna(r["anova_p"]) else np.nan
                        )
                    })

    return pd.DataFrame(rows)


# =============================================================================
# PLOTS
# =============================================================================
def save_boxplots(df):
    plot_dir = os.path.join(OUTPUT_DIR, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    order_noise = NOISE_ORDER
    order_status = get_present_status_order(df)

    for feature in ["TPV22", "RMSSD"]:
        sub = df[np.isfinite(df[feature])].copy()

        plt.figure(figsize=(12, 6))
        sns.boxplot(
            data=sub,
            x="noise",
            y=feature,
            hue="status_name",
            order=order_noise,
            hue_order=order_status
        )
        plt.title(f"{feature} distribution by condition across noise levels")
        plt.xlabel("Noise level")
        plt.ylabel(feature)
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f"boxplot_{feature}_by_noise.png"), dpi=300)
        plt.close()

    fig, axes = plt.subplots(1, len(order_noise), figsize=(5.5 * len(order_noise), 5), sharey=False)
    for i, noise_name in enumerate(order_noise):
        ax = axes[i]
        sub = df[(df["noise"] == noise_name) & np.isfinite(df["TPV22"])].copy()
        sns.boxplot(
            data=sub,
            x="status_name",
            y="TPV22",
            order=order_status,
            ax=ax
        )
        ax.set_title(f"TPV22 - {noise_name}")
        ax.set_xlabel("")
        ax.set_ylabel("TPV22")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "boxplot_TPV22_panels.png"), dpi=300)
    plt.close()

    fig, axes = plt.subplots(1, len(order_noise), figsize=(5.5 * len(order_noise), 5), sharey=False)
    for i, noise_name in enumerate(order_noise):
        ax = axes[i]
        sub = df[(df["noise"] == noise_name) & np.isfinite(df["RMSSD"])].copy()
        sns.boxplot(
            data=sub,
            x="status_name",
            y="RMSSD",
            order=order_status,
            ax=ax
        )
        ax.set_title(f"RMSSD - {noise_name}")
        ax.set_xlabel("")
        ax.set_ylabel("RMSSD")
    plt.tight_layout()
    plt.savefig(os.path.join(plot_dir, "boxplot_RMSSD_panels.png"), dpi=300)
    plt.close()


def save_robustness_lineplots(stats_df):
    plot_dir = os.path.join(OUTPUT_DIR, "plots")
    os.makedirs(plot_dir, exist_ok=True)

    x_order = NOISE_ORDER

    for metric in ["anova_p", "eta_sq", "consistency_ratio"]:
        plt.figure(figsize=(8, 5))
        for feature in ["TPV22", "RMSSD"]:
            sub = stats_df[stats_df["feature"] == feature].copy()
            sub["noise"] = pd.Categorical(sub["noise"], categories=x_order, ordered=True)
            sub = sub.sort_values("noise")
            plt.plot(sub["noise"].astype(str), sub[metric], marker="o", label=feature)

        plt.title(f"Robustness comparison: {metric}")
        plt.xlabel("Noise level")
        plt.ylabel(metric)
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(plot_dir, f"robustness_{metric}.png"), dpi=300)
        plt.close()


# =============================================================================
# MAIN
# =============================================================================
if __name__ == "__main__":
    df_all = load_all_preextracted_csvs()

    if len(df_all) == 0:
        raise SystemExit("[WARN] No data available.")

    print("\n[INFO] Final merged shape:", df_all.shape)
    print("[INFO] Final status dist:", df_all["status_name"].value_counts(dropna=False).to_dict())
    print("[INFO] Final noise dist:", df_all["noise"].value_counts(dropna=False).to_dict())

    raw_csv = os.path.join(OUTPUT_DIR, "TPV22_RMSSD_all_windows_with_noise.csv")
    df_all.to_csv(raw_csv, index=False)
    print(f"[INFO] Saved merged feature CSV: {raw_csv}")

    for noise_name in NOISE_ORDER:
        df_sub = df_all[df_all["noise"] == noise_name].copy()
        save_path = os.path.join(OUTPUT_DIR, f"features_{noise_name}.csv")
        df_sub.to_csv(save_path, index=False)
        print(f"[INFO] Saved: {save_path}")

    stats_tpv = summarize_by_noise_and_feature(df_all, "TPV22")
    stats_rmssd = summarize_by_noise_and_feature(df_all, "RMSSD")
    stats_all = pd.concat([stats_tpv, stats_rmssd], ignore_index=True)

    stats_csv = os.path.join(OUTPUT_DIR, "noise_robustness_stats.csv")
    stats_all.to_csv(stats_csv, index=False)
    print(f"[INFO] Saved stats CSV: {stats_csv}")

    table12_df = build_table12_style(stats_all)
    table12_csv = os.path.join(OUTPUT_DIR, "table12_style_summary.csv")
    table12_df.to_csv(table12_csv, index=False)
    print(f"[INFO] Saved table-style CSV: {table12_csv}")

    save_boxplots(df_all)
    save_robustness_lineplots(stats_all)
    print(f"[INFO] Saved plots under: {os.path.join(OUTPUT_DIR, 'plots')}")

    print("\n===== ANOVA / Robustness Summary =====")
    print(stats_all.to_string(index=False))

    print("\n===== TABLE STYLE SUMMARY =====")
    print(table12_df.to_string(index=False))


[INFO] Loading noise=orig
       TPV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP_TPV/noQC_BVP.csv
       HRV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV/EmpaticaE4Stress_IBI_HRV_1min.csv
[INFO] merged shape (orig): (900, 13)
[INFO] status dist (orig): {'stress': 644, 'normal': 256}

[INFO] Loading noise=alpha_0.01
       TPV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP-TPV-noise/noQC_BVP_alpha_0.01.csv
       HRV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV-noise/EmpaticaE4Stress_BVP_HRV_1min_alpha_0.01.csv
[INFO] merged shape (alpha_0.01): (1035, 13)
[INFO] status dist (alpha_0.01): {'stress': 671, 'normal': 364}

[INFO] Loading noise=alpha_0.03
       TPV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/BVP-TPV-noise/noQC_BVP_alpha_0.03.csv
       HRV: /content/drive/MyDrive/Colab Notebooks/BM/TPV-stress/EmpaticaE4Stress/HRV-noise/EmpaticaE4